In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from catboost import CatBoostRegressor
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import PowerTransformer
import category_encoders as ce


/Users/katiegalaeva/miniconda3/envs/catboost_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
train = pd.read_csv('train.csv')
train

,name,_id,host_name,location_cluster,location,lat,lon,type_house,sum,min_days,amt_reviews,last_dt,avg_reviews,total_host,target
0,Belle Harbor 4 BR 2 bath- 1 bl from Beach,40327248,Sarina,Queens,Neponsit,40.57215,-73.85822,Entire home/apt,350,2,5,2019-07-07,2.88,1,334
1,"Come see Brooklyn, New York",13617520,Howard T.,Brooklyn,Clinton Hill,40.69172,-73.96934,Shared room,40,5,8,2015-02-25,0.13,1,0
2,Large 2Br on W71st & Columbus Feb 19-28,26754726,Julie,Manhattan,Upper West Side,40.77673,-73.98011,Entire home/apt,200,5,0,NaN,NaN,1,0
3,Perfect bedroom. Near Subways Columbia CityCol...,16721721,Federico,Manhattan,Harlem,40.81530,-73.95080,Private room,65,2,18,2018-11-04,0.64,1,0
4,Cozy Sun Filled Fresh Guest Room in Artsy Bush...,22246463,Lisa,Brooklyn,Bushwick,40.70230,-73.92935,Private room,99,2,26,2019-06-23,0.76,1,155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36666,Manhattan Apartment - Near North Central Park,108286255,Gabrielle,Manhattan,Harlem,40.81264,-73.94231,Private room,60,1,3,2019-01-01,0.46,1,0
36667,Elegant 1876 Boerum Hill Brownstone,16395150,Patti,Brooklyn,Boerum Hill,40.68554,-73.98471,Entire home/apt,265,2,10,2016-09-07,0.18,1,0
36668,Super cute and sunny 2 bedroom,39304,Andrea,Brooklyn,Williamsburg,40.71852,-73.94165,Entire home/apt,240,365,0,NaN,NaN,1,363
36669,TWO BIG Rooms - a Park Slope Oasis,15942513,Tom,Brooklyn,South Slope,40.66617,-73.98862,Private room,90,2,15,2019-06-18,2.34,1,91


In [14]:
def preprocess_data(df):
    df['host_name'].fillna('unknown', inplace=True)
    df['amt_reviews'].fillna(0, inplace=True)
    df['avg_reviews'].fillna(df['avg_reviews'].median(), inplace=True)
    

    #df['target'] = np.log1p(df['target'])  # Отмена через np.expm1()
    
    df.drop(['_id', 'name', 'last_dt'], axis=1, inplace=True)
    
    return df

In [15]:
def add_features(df):

    from sklearn.cluster import KMeans
    coords = df[['lat', 'lon']].dropna()
    kmeans = KMeans(n_clusters=20, random_state=42).fit(coords)
    df['location_cluster_kmeans'] = kmeans.predict(df[['lat', 'lon']])
    

    df['host_listings_percent'] = df['total_host'] / df['total_host'].max()
    
    df['price_per_day'] = df['sum'] / (df['min_days'] + 1)
    
    return df

In [16]:
train = preprocess_data(train)

/var/folders/rm/c6dtgnt92mncnb60whf8_fqm0000gn/T/ipykernel_40205/3281729639.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['host_name'].fillna('unknown', inplace=True)
/var/folders/rm/c6dtgnt92mncnb60whf8_fqm0000gn/T/ipykernel_40205/3281729639.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alway

In [17]:
train = add_features(train)

In [18]:
train = train.drop(columns=['host_listings_percent'])
train = train.drop(columns=[ 'sum'])

In [66]:
train

,host_name,location_cluster,location,lat,lon,type_house,min_days,amt_reviews,avg_reviews,total_host,target,location_cluster_kmeans,price_per_day
0,Sarina,Queens,Neponsit,40.57215,-73.85822,Entire home/apt,2,5,2.88,1,334,13,116.666667
1,Howard T.,Brooklyn,Clinton Hill,40.69172,-73.96934,Shared room,5,8,0.13,1,0,7,6.666667
2,Julie,Manhattan,Upper West Side,40.77673,-73.98011,Entire home/apt,5,0,0.72,1,0,11,33.333333
3,Federico,Manhattan,Harlem,40.81530,-73.95080,Private room,2,18,0.64,1,0,3,21.666667
4,Lisa,Brooklyn,Bushwick,40.70230,-73.92935,Private room,2,26,0.76,1,155,2,33.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36666,Gabrielle,Manhattan,Harlem,40.81264,-73.94231,Private room,1,3,0.46,1,0,3,30.000000
36667,Patti,Brooklyn,Boerum Hill,40.68554,-73.98471,Entire home/apt,2,10,0.18,1,0,7,88.333333
36668,Andrea,Brooklyn,Williamsburg,40.71852,-73.94165,Entire home/apt,365,0,0.72,1,363,9,0.655738
36669,Tom,Brooklyn,South Slope,40.66617,-73.98862,Private room,2,15,2.34,1,91,7,30.000000


In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import PowerTransformer
import category_encoders as ce


df = train.copy()
X = df.drop(columns=['target'])
y = df['target']


In [20]:

categorical_columns = ['host_name',	'location_cluster',	'location', 'type_house','location_cluster_kmeans'] 
X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

target_encoder = ce.TargetEncoder(cols=categorical_columns)
X_train_encoded = target_encoder.fit_transform(X_train_full, y_train)
X_test_encoded = target_encoder.transform(X_test_full)


In [11]:
X_train_encoded

,name,_id,host_name,location_cluster,location,lat,lon,type_house,min_days,amt_reviews,last_dt,avg_reviews,total_host,location_cluster_kmeans,price_per_day
14881,Central & Comfy East Village Studio,225620034,141.683577,111.517520,77.605263,40.73011,-73.98405,111.215533,1,13,2019-07-02,3.07,1,83.467213,94.500000
36476,MILES DAVIS BIG BEAUTIFUL BEDROOM,203982404,112.786132,111.517520,98.558699,40.80822,-73.94051,110.917212,2,20,2019-06-24,3.09,6,99.766678,30.000000
6906,Quiet Haven in Park Slope Huge 2BR,316474,100.403280,100.024342,87.085380,40.66693,-73.98801,111.215533,2,66,2019-01-02,1.13,1,86.122507,75.000000
16418,Perfect 1 BR in UWS,22157668,120.928209,111.517520,96.337716,40.78074,-73.98393,111.215533,5,0,NaN,NaN,1,131.492676,41.666667
27440,East Village Floor Thru plus Garden,7245581,88.942387,111.517520,77.605263,40.72251,-73.97677,111.215533,100,5,2019-01-02,0.10,19,83.467213,0.861386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16850,Beautiful Bedroom NEAR SUBWAYS 25 min to Manha...,139490165,94.971138,100.024342,95.337550,40.68469,-73.90794,110.917212,2,9,2018-06-03,0.41,3,109.701742,36.666667
6265,"Sunny room in historic Bedstuy, Brooklyn",124171,96.424160,100.024342,113.325853,40.68533,-73.94231,110.917212,1,1,2018-07-08,0.08,1,101.545297,30.000000
11284,Artsy 1 bedroom Apartment in East Village,15537429,105.001707,111.517520,77.605263,40.72785,-73.97976,111.215533,1,80,2019-06-26,4.62,3,83.467213,77.500000
860,Private Tent - Shared Room (Bed-2),44620317,102.695681,143.003239,217.661276,40.73977,-73.85777,163.344778,1,34,2018-09-21,0.76,4,151.588710,18.500000


In [56]:
y_binary_train = (y_train > 0).astype(int)
y_binary_test  = (y_test > 0).astype(int)

clf = CatBoostClassifier(
    verbose=0,
    cat_features=categorical_columns,
    random_state=42
)

param_grid_classifier = {
    'iterations': [800, 1000],
    'learning_rate': [0.07, 0.05],
    'depth': [6, 8, 10],
    'l2_leaf_reg': [3, 5]
}


grid_clf = GridSearchCV(clf, param_grid_classifier, cv=3, scoring='accuracy', n_jobs=-1)
grid_clf.fit(X_train_full, y_binary_train)

print("Best CatBoostClassifier parameters:", grid_clf.best_params_)
print("Best CV accuracy (classifier):", grid_clf.best_score_)

# Используем лучшую модель для предсказания на тестовой выборке
best_clf = grid_clf.best_estimator_
y_pred_binary = best_clf.predict(X_test_full)
print("CatBoost Classifier Test Accuracy:", accuracy_score(y_binary_test, y_pred_binary))

Best CatBoostClassifier parameters: {'depth': 6, 'iterations': 1000, 'l2_leaf_reg': 5, 'learning_rate': 0.07}
Best CV accuracy (classifier): 0.8180732519630866
CatBoost Classifier Test Accuracy: 0.8152692569870484


In [ ]:
mask_positive_train = (y_train > 0)
X_train_reg = X_train_full[mask_positive_train]
y_train_reg = y_train[mask_positive_train]


pt = PowerTransformer(method='yeo-johnson', standardize=False)
y_train_reg_transformed = pt.fit_transform(y_train_reg.to_numpy().reshape(-1, 1)).ravel()

mask_positive_test = (y_test > 0)
X_test_reg = X_test_full[mask_positive_test]
y_test_reg = y_test[mask_positive_test]

reg = CatBoostRegressor(
    verbose=0,
    cat_features=categorical_columns, 
)

param_grid = {
    'iterations': [800, 500],
    'learning_rate': [0.08, 0.05],
    'depth': [6, 8, 10],
    'l2_leaf_reg': [7, 5]
}

grid_reg = GridSearchCV(reg, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_reg.fit(X_train_reg, y_train_reg_transformed)

print("Best parameters:", grid_reg.best_params_)
print("Best CV score (neg MSE):", grid_reg.best_score_)


best_reg = grid_reg.best_estimator_

y_pred_reg_transformed = best_reg.predict(X_test_reg)

y_pred_reg = pt.inverse_transform(y_pred_reg_transformed.reshape(-1, 1)).ravel()
y_test_reg_orig = pt.inverse_transform(y_test_reg.to_numpy().reshape(-1, 1)).ravel()

rmse = np.sqrt(mean_squared_error(y_test_reg_orig, y_pred_reg))
print("CatBoost Regressor RMSE (original scale):", rmse)

Best parameters: {'depth': 8, 'iterations': 800, 'l2_leaf_reg': 5, 'learning_rate': 0.05}
Best CV score (neg MSE): -100.77835768141712
CatBoost Regressor RMSE (original scale): 13292.456426868845


In [57]:
def predict_target(X_new):

    pred_binary = best_clf.predict(X_new)
    preds = np.zeros(len(X_new))

    idx_positive = np.where(pred_binary.reshape(-1) == 1)[0]
    if len(idx_positive) > 0:
        X_new_pos = X_new.iloc[idx_positive]
        pred_reg_transformed = best_reg.predict(X_new_pos)
        pred_reg = pt.inverse_transform(pred_reg_transformed.reshape(-1, 1)).ravel()
        preds[idx_positive] = pred_reg
    return preds

In [ ]:
test = pd.read_csv('test.csv')

def preprocess_data_test(df):
   
    df['host_name'].fillna('unknown', inplace=True)
    df['amt_reviews'].fillna(0, inplace=True)
    df['avg_reviews'].fillna(df['avg_reviews'].median(), inplace=True)
    

    df.drop(['_id', 'name', 'last_dt'], axis=1, inplace=True)
    
    return df

test = preprocess_data_test(test)
test = add_features(test)

/var/folders/rm/c6dtgnt92mncnb60whf8_fqm0000gn/T/ipykernel_40205/494725681.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['host_name'].fillna('unknown', inplace=True)
/var/folders/rm/c6dtgnt92mncnb60whf8_fqm0000gn/T/ipykernel_40205/494725681.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always 

In [59]:
test = test.drop(columns=['host_listings_percent'])
test = test.drop(columns=[ 'sum'])

In [60]:
y_new_pred = predict_target(test)

In [61]:
y_new_pred

array([220.47659452, 216.68321771, 305.12657014, ..., 260.03011774,
         0.        , 161.26466834])

In [62]:
submission = pd.DataFrame({'prediction': y_new_pred })

submission = submission.reset_index()

submission.rename(columns={'index': 'index', 'prediction': 'prediction'}, inplace=True)

submission.to_csv("two_model_catboost_3.csv", index=False)